# 34 - Reading the dead-time flip: what to set in the ASIAIR

**Purpose.** Explain what notebook `33` found, for someone deciding how often to dither. `33` is
written for someone *checking* it; this one is slow on purpose.

It explains four things: why there is a flip at all; where it sits; what it means in "dither every
k frames"; and what dithering less buys even when the gain stays at 50.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `dither_flip.csv` and `dither_flip_constants.json`, with the model constants `33` used.
If anything here disagrees with `results/`, `results/` is right and this notebook is the bug.

**It assumes** `32_tradeoff_curve_read.ipynb`, which explains the curve and why dead time decides
the gain.

**The headline, recomputed below:** gain 200 beats gain 50 only once dead time falls under about
**3.4 s** per sub. Dithering alone costs about 30 s, so that means dithering every 12th frame or
rarer, *and* a download near the archive's 0.68 s. On our own night the download can only be shown to be
under 8 s, and 8 s is already past the flip. **Keep gain 50.**

In [ ]:
import json
import pathlib
import sys

import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import model as M

RESULTS = pathlib.Path.cwd().parent / "results"
flip = pd.read_csv(RESULTS / "dither_flip.csv")
dc = M.Constants.load(RESULTS / "dither_flip_constants.json")
sky = json.loads((RESULTS / "sky_constants.json").read_text())
T_DEAD = json.loads((RESULTS / "model_constants.json").read_text())["t_dead"]["value"]
DECOMP = sky["t_dead_decomposition"]["value"]
print(dc.entry("t_dead_flip")["note"])
print()
print(dc.entry("dither_cadence_flip")["note"].split("  model predictions")[0])

## 1. Why there is a flip at all

`32` split the night's SNR into two factors: the share of the night the shutter is **exposing**,
and the share of the noise that is **sky** rather than read noise. At equal clipping, gain 200
needs subs about six times shorter than gain 50. It wins the sky factor, since its read noise is
about one electron. It loses the exposing factor, because every sub pays the same dead time.

As dead time shrinks, the exposing factor goes to 1 for both gains and only the sky factor is left,
and gain 200 wins that one. So there has to be a dead time where they cross. Here is the ratio at
1% of stars clipped, read from `dither_flip.csv`:

In [ ]:
one = flip[flip.budget == 0.01].set_index("t_dead")
pick = [0.0, 1.0, 2.0, 3.0, 3.4, 3.5, 4.0, 6.0, 8.0, 12.0, 16.0, 24.0, T_DEAD]
print(one.loc[one.index.isin([round(p, 6) for p in pick]),
              ["gain", "t", "n_subs", "ratio_200_50", "snr_vs_today"]].round(3).to_string())

**The crossing is shallow on both sides.** At zero dead time, gain 200 wins by less than 8%. At
today's 31.5 s, it loses by about 27%. The gain choice is lopsided in one direction only: getting it
wrong today costs a quarter of the SNR, while getting it "right" at zero dead time buys a few
percent.

## 2. Where it sits, at each share of clipped stars

In [ ]:
for budget, v in dc.entry("t_dead_flip")["value"].items():
    print(f"{budget:>4}% clipped   gain 200: {v['200_vs_50']}   gain 100: {v['100_vs_50']}")
print("\nmost the higher gain can ever win by (t_dead = 0):")
for budget, v in dc.entry("ratio_at_zero_dead_time")["value"].items():
    print(f"{budget:>4}% clipped   gain 200: {v['200_vs_50']}   gain 100: {v['100_vs_50']}")

**From 1% to 5% clipped, the flip is the same 3.3-3.4 s.** That is the useful range of the curve,
where `32` puts the choices worth making. At 0.5% gain 200 has no sub short enough on the
10-600 s grid; and gain 100 never beats gain 50 there at any dead time.

**Ignore the 10% row.** Gain 50's best sub there is 600 s, the longest on `31`'s grid, and it clips
only 5.8% of stars: it cannot spend the budget, because the grid stops. Gain 100 wins only because
at the same 600 s it has less read noise. That is the grid's edge, not the physics, and `33` says so
in its note. Nobody chasing star colour shoots there, and nobody has shown the mount tracks 600 s.

## 3. From seconds to the ASIAIR

Dithering every `k`-th frame spreads the settle over `k` subs:
`t_dead(k) = download + settle / k`. Only the total was measured. The split is inferred, and the
record holds two downloads:

- **0.68 s**, from the archive, on frames that skipped a dither. Not ours.
- **7.99 s**, the shortest gap on our own night. An upper bound, and the only figure we can prove.

The one check available is the archive itself, which dithered every second frame:

In [ ]:
for name, dl in (("archive", DECOMP["download_and_save_archive"]),
                 ("bound", DECOMP["download_and_save_bound"])):
    print(f"download {dl:.2f} s: every 2nd frame -> {dl + (T_DEAD - dl) / 2:.1f} s   "
          "(the archive measured 17-20 s)")
print()
for budget, v in dc.entry("dither_cadence_flip")["value"].items():
    print(f"{budget:>4}% clipped   gain 200 needs dithering every: {v['200_vs_50']}")

**Under the archive's download, gain 200 wins once dithering is every 12th frame or rarer.** Under our own bound it
never does: 7.99 s of download is already more than the 3.4 s flip, so no cadence reaches it.

**And the archive check leans toward the bound.** Every second frame gives 16.1 s with the
archive's download and 19.7 s with ours; the archive measured 17-20 s. That is not proof - the
archive's settle may have been different - but it does not support the fast download. The flip is
probably out of reach on this rig.

## 4. What dithering less buys, at gain 50

The gain question has a small answer. The bigger one is dead time itself: every second a sub does
not pay is a second of exposing, whichever gain wins. `snr_vs_today` is the best setting at each
budget against the best at today's 31.5 s.

In [ ]:
rows = []
for name, dl in (("archive", DECOMP["download_and_save_archive"]),
                 ("bound", DECOMP["download_and_save_bound"])):
    for k in (1, 2, 4, 8):
        td = dl + (T_DEAD - dl) / k
        near = flip.t_dead.iloc[(flip.t_dead - td).abs().argmin()]
        for b in (0.01, 0.02, 0.05):
            r = flip[(flip.t_dead == near) & (flip.budget == b)].iloc[0]
            rows.append({"download": name, "dither every": k, "t_dead, s": round(td, 1),
                         "clipped %": 100 * b, "gain": int(r.gain), "t, s": r.t,
                         "SNR vs today": round(r.snr_vs_today, 3)})
print(pd.DataFrame(rows).to_string(index=False))

**Dithering every 4th frame instead of every frame buys 7-10% SNR at 1% clipped**, under either
download, with the gain still at 50. At 5% clipped it buys about 2%, because those subs are already
long enough that dead time is a small share of the night. So the cadence matters most exactly where
star colour matters most: short subs.

**That gain is real only if stacking does not lose it back.** This is the one thing `33` does not
price, and it is the next section.

## 5. What this was drawn under, and which way each pushes

- **No stacking loss at any cadence.** Stacking loss was measured only with dithering every frame,
  and only to 18 subs. Fewer dither positions give outlier rejection less to work with and let fixed
  pattern line up between subs. That pushes **against dithering less** - so every gain in section 4
  is the most it could be - and **against gain 200 most of all**, since it needs the biggest stacks.
  Every flip here is the most favourable case for the higher gain.
- **The download is not measured.** Our night dithered after every frame by design, so no gap on it
  shows a download alone. The answer to "is the flip reachable" hangs on this one number.
- **One night, one field**, as for `31`.

## 6. What is settled, and what is next

**Settled.**
- **On this rig, gain 50 stays the setting** at every share of clipped stars from 0.5% to 5%, at any
  dither cadence the measured download allows. Gain 200 needs dead time under 3.4 s, and saving a
  frame cannot be shown to take less than 8 s.
- **Even where gain 200 wins, it wins by less than 8%.** The gain is not a lever worth pulling for
  this.
- **Dead time is the lever.** Dithering less often buys up to about 10% SNR at gain 50, more than
  any gain change, *if* stacking holds up.

**Not settled, named rather than fixed.**
1. **The download alone.** A few frames with dithering off, timestamps read, gives it directly.
   That says whether the flip is reachable at all, and which row of section 4 is real.
2. **Stacking loss at a lower cadence.** Whether dithering every 4th frame keeps its 7-10% after
   rejection. This needs a night, not arithmetic.
3. **How long the mount tracks**, as for `32`.